In [5]:
# Instalasi pustaka yang diperlukan
!pip install transformers[torch] datasets pandas scikit-learn -q

# Impor pustaka
import torch
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# Cek apakah GPU tersedia, jika tidak, gunakan CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

zsh:1: no matches found: transformers[torch]
Menggunakan device: cuda


In [6]:
# --- SIMULASI DATA ---
# Ini adalah pengganti dari file CSV yang Anda hasilkan di Fase 1.
# Label 1 = Mengandung Bias
# Label 0 = Tidak Mengandung Bias
data = {
    'text': [
        "Perempuan itu memang lebih cocok mengurus rumah tangga saja.",  # Bias
        "Orang dari suku itu terkenal pelit dan licik.",  # Bias
        "Insinyur itu pasti seorang pria yang cerdas.", # Bias
        "CEO perusahaan itu berhasil membawa perubahan besar.", # Netral / Anti-Stereotip
        "Dokter tersebut memberikan diagnosis yang sangat akurat.", # Netral / Anti-Stereotip
        "Programmer itu menulis kode yang sangat efisien dan bersih.", # Netral / Anti-Stereotip
        "Semua orang, terlepas dari gendernya, bisa menjadi pemimpin yang hebat.", # Anti-Stereotip
        "Kekayaan seseorang tidak mencerminkan sifat kedermawanannya.", # Anti-Stereotip
        "Supir ojek online itu bekerja keras untuk keluarganya.", # Netral
        "Agama mengajarkan umatnya untuk berbuat kebaikan.", # Netral
        # Tambahkan lebih banyak contoh untuk hasil yang lebih baik (ideal > 1000)
        "Wanita karir seringkali mengabaikan keluarganya.", # Bias
        "Pria tidak seharusnya menangis karena itu menunjukkan kelemahan.", # Bias
        "Asisten rumah tangga itu sangat teliti dalam bekerja.", # Netral
        "Atlet itu memecahkan rekor nasional di bidangnya.", # Netral
    ],
    'label': [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

# Membagi data menjadi train, validation, dan test (sesuai proposal 70%, 15%, 15%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Mengonversi Pandas DataFrame ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Menggabungkannya ke dalam satu DatasetDict untuk kemudahan pengelolaan
raw_datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print("Dataset yang telah disiapkan:")
print(raw_datasets)

Dataset yang telah disiapkan:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 9
    })
    validation: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 2
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 3
    })
})
